In [ ]:
!pip install chromadb rank-bm25 sentence-transformers

In [ ]:
import re
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

In [ ]:

# ----------------------------
# 1. Documents
# ----------------------------

documents = [
    "The admin password for the firewall is Admin!@#2026.",
    "Our backend team uses Python and Go for scalable microservices.",
    "Canine influenza is spreading rapidly among domestic pets.",
    "The support desk phone line is available at extension 9021.",
    "RAG systems often combine dense vector search with sparse BM25 search.",
]

doc_ids = [f"doc_{i}" for i in range(len(documents))]


# ----------------------------
# 2. Tokenizer for BM25
# ----------------------------

def tokenize(text: str):
    return re.findall(r"\b\w+\b|[^\w\s]", text.lower())


# ----------------------------
# 3. Build BM25 index
# ----------------------------

tokenized_docs = [tokenize(doc) for doc in documents]
bm25 = BM25Okapi(tokenized_docs)


In [ ]:
tokenized_docs

[['the',
  'admin',
  'password',
  'for',
  'the',
  'firewall',
  'is',
  'admin',
  '!',
  '@',
  '#',
  '2026',
  '.'],
 ['our',
  'backend',
  'team',
  'uses',
  'python',
  'and',
  'go',
  'for',
  'scalable',
  'microservices',
  '.'],
 ['canine',
  'influenza',
  'is',
  'spreading',
  'rapidly',
  'among',
  'domestic',
  'pets',
  '.'],
 ['the',
  'support',
  'desk',
  'phone',
  'line',
  'is',
  'available',
  'at',
  'extension',
  '9021',
  '.'],
 ['rag',
  'systems',
  'often',
  'combine',
  'dense',
  'vector',
  'search',
  'with',
  'sparse',
  'bm25',
  'search',
  '.']]

In [ ]:

# ----------------------------
# 4. Build ChromaDB vector index
# ----------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.EphemeralClient()

collection = chroma_client.create_collection(
    name="hybrid_rag_demo"
)

embeddings = embedding_model.encode(documents).tolist()

collection.add(
    ids=doc_ids,
    documents=documents,
    embeddings=embeddings,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

# ----------------------------
# 5. Vector search
# ----------------------------

def vector_search(query: str, top_k: int = 5):
    query_embedding = embedding_model.encode([query]).tolist()[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )

    ranked = []

    for doc_id, doc, distance in zip(
        results["ids"][0],
        results["documents"][0],
        results["distances"][0],
    ):
        ranked.append({
            "id": doc_id,
            "text": doc,
            "distance": distance,
        })

    return ranked


In [ ]:
# ----------------------------
# 6. BM25 search
# ----------------------------

def bm25_search(query: str, top_k: int = 5):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )[:top_k]

    ranked = []

    for i in ranked_indices:
        ranked.append({
            "id": doc_ids[i],
            "text": documents[i],
            "score": float(scores[i]),
        })

    return ranked


In [ ]:

# ----------------------------
# 7. Reciprocal Rank Fusion
# ----------------------------

def reciprocal_rank_fusion(result_lists, k: int = 60):
    """
    RRF score:
        score(d) = sum over rankers of 1 / (k + rank)

    rank is 1-based.
    k controls how much lower-ranked documents are dampened.
    Common default: k = 60.
    """

    fused_scores = {}
    doc_lookup = {}

    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item["id"]

            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0.0
                doc_lookup[doc_id] = item["text"]

            fused_scores[doc_id] += 1.0 / (k + rank)

    fused_ranked = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    return [
        {
            "id": doc_id,
            "text": doc_lookup[doc_id],
            "rrf_score": score,
        }
        for doc_id, score in fused_ranked
    ]


In [ ]:

# ----------------------------
# 8. Hybrid retrieval
# ----------------------------

def hybrid_search(query: str, vector_k: int = 5, bm25_k: int = 5, final_k: int = 3):
    vector_results = vector_search(query, top_k=vector_k)
    bm25_results = bm25_search(query, top_k=bm25_k)

    fused_results = reciprocal_rank_fusion(
        [vector_results, bm25_results],
        k=60,
    )

    return {
        "query": query,
        "vector_results": vector_results,
        "bm25_results": bm25_results,
        "hybrid_results": fused_results[:final_k],
    }


In [ ]:
queries = [
    "Tell me about sick dogs",
    "What is the firewall password Admin!@#2026?",
    "How do RAG systems combine semantic and keyword search?",
]

for query in queries:
    result = hybrid_search(query)

    print("\n" + "=" * 80)
    print("QUERY:", result["query"])

    print("\nVECTOR RESULTS")
    for r in result["vector_results"]:
        print(f'{r["id"]}: {r["text"]}')

    print("\nBM25 RESULTS")
    for r in result["bm25_results"]:
        print(f'{r["id"]}: {r["text"]} | BM25={r["score"]:.4f}')

    print("\nHYBRID RRF RESULTS")
    for r in result["hybrid_results"]:
        print(f'{r["id"]}: {r["text"]} | RRF={r["rrf_score"]:.6f}')


QUERY: Tell me about sick dogs

VECTOR RESULTS
doc_2: Canine influenza is spreading rapidly among domestic pets.
doc_1: Our backend team uses Python and Go for scalable microservices.
doc_4: RAG systems often combine dense vector search with sparse BM25 search.
doc_3: The support desk phone line is available at extension 9021.
doc_0: The admin password for the firewall is Admin!@#2026.

BM25 RESULTS
doc_0: The admin password for the firewall is Admin!@#2026. | BM25=0.0000
doc_1: Our backend team uses Python and Go for scalable microservices. | BM25=0.0000
doc_2: Canine influenza is spreading rapidly among domestic pets. | BM25=0.0000
doc_3: The support desk phone line is available at extension 9021. | BM25=0.0000
doc_4: RAG systems often combine dense vector search with sparse BM25 search. | BM25=0.0000

HYBRID RRF RESULTS
doc_2: Canine influenza is spreading rapidly among domestic pets. | RRF=0.032266
doc_1: Our backend team uses Python and Go for scalable microservices. | RRF=0.0322

In [13]:
hybrid_search("tell me about the desk phone")

{'query': 'tell me about the desk phone',
 'vector_results': [{'id': 'doc_3',
   'text': 'The support desk phone line is available at extension 9021.',
   'distance': 1.0170674324035645},
  {'id': 'doc_1',
   'text': 'Our backend team uses Python and Go for scalable microservices.',
   'distance': 1.7711941003799438},
  {'id': 'doc_4',
   'text': 'RAG systems often combine dense vector search with sparse BM25 search.',
   'distance': 1.9325435161590576},
  {'id': 'doc_0',
   'text': 'The admin password for the firewall is Admin!@#2026.',
   'distance': 1.9597407579421997},
  {'id': 'doc_2',
   'text': 'Canine influenza is spreading rapidly among domestic pets.',
   'distance': 1.9969381093978882}],
 'bm25_results': [{'id': 'doc_3',
   'text': 'The support desk phone line is available at extension 9021.',
   'score': 2.554221810650157},
  {'id': 'doc_0',
   'text': 'The admin password for the firewall is Admin!@#2026.',
   'score': 0.45706355975228435},
  {'id': 'doc_1',
   'text': 'Our

### Add the Agent Loop to have Agentic Hybrid RAG

In [17]:
from openai import OpenAI
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=OPENAI_API_KEY
)

MODEL = "gpt-4.1-mini"

In [18]:
# =========================================================
# OPENAI GENERATION FUNCTION
# =========================================================

def generate(prompt):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content

In [31]:
# =========================================================
# AGENT HELPERS
# =========================================================

MAX_ITERATIONS = 1


def build_context(results, top_k=3):

    context = []

    for i, r in enumerate(results[:top_k], start=1):
        context.append(f"[{i}] {r['text']}")

    return "\n".join(context)


def should_continue(response_text):

    trigger_phrases = [
        "not enough information",
        "insufficient context",
        "need more information",
        "missing information",
        "unclear",
    ]

    response_lower = response_text.lower()

    return any(p in response_lower for p in trigger_phrases)


def rewrite_query(original_query, previous_answer):

    prompt = f"""
User query:
{original_query}

Current incomplete answer:
{previous_answer}

Rewrite the query to retrieve missing information.

Only output the rewritten query.
"""

    return generate(prompt).strip()

In [29]:
# =========================================================
# AGENTIC RAG LOOP
# =========================================================

def agentic_rag(query):

    current_query = query

    memory = []

    for iteration in range(MAX_ITERATIONS):

        print(f"\n===== ITERATION {iteration+1} =====")
        print("QUERY:", current_query)

        # -----------------------------------
        # HYBRID RETRIEVAL
        # -----------------------------------

        retrieved_docs = hybrid_search(
            current_query,
            vector_k=5,
            bm25_k=5,
            final_k=5,
        )["hybrid_results"]

        context = build_context(retrieved_docs)

        print("\nRETRIEVED CONTEXT:\n")
        print(context)

        # -----------------------------------
        # GENERATION
        # -----------------------------------

        prompt = f"""
You are a grounded RAG assistant.

Rules:
- Answer ONLY from retrieved context
- Do not hallucinate
- If context is insufficient say:
  "Not enough information."

Question:
{query}

Retrieved Context:
{context}

Answer:
"""

        response = generate(prompt)

        print("\nMODEL RESPONSE:\n")
        print(response)

        memory.append({
            "query": current_query,
            "context": context,
            "response": response,
        })

        # -----------------------------------
        # STOP CONDITION
        # -----------------------------------

        if not should_continue(response):

            return {
                "final_answer": response,
                "trajectory": memory,
            }

        # -----------------------------------
        # QUERY REWRITE
        # -----------------------------------

        current_query = rewrite_query(
            original_query=query,
            previous_answer=response,
        )

        print("\nREWRITTEN QUERY:\n")
        print(current_query)

    return {
        "final_answer": response,
        "trajectory": memory,
    }

In [ ]:
result = agentic_rag(
    "Tell me about the desk phone"
)

print("\n\n==============================")
print("FINAL ANSWER")
print("==============================\n")

print(result["final_answer"])

In [22]:
for i, step in enumerate(result["trajectory"], start=1):

    print("\n" + "="*80)
    print(f"STEP {i}")

    print("\nQUERY:")
    print(step["query"])

    print("\nCONTEXT:")
    print(step["context"])

    print("\nRESPONSE:")
    print(step["response"])

KeyError: 'trajectory'